In [ ]:
# подключаем базовые библиотеки
import copy
import numpy as np
import networkx as nx
import plotly.graph_objects as go


# параметры задачи
initial_budget = 10000
contract_cost_per_neighbor = 300
income_per_affected = 50
max_contracts_per_day = 10
campaign_duration = 60
threshold = 0.18

In [ ]:
#удалите следующую строку, если запускаете код не в браузере
!gdown --id 1CgegcarUDFN8jikjdtCga2jZ93NB99nL

# считали граф из файла в граф библиотеки networkx
G = nx.read_edgelist('marketing_edges.txt')

# сделали id узлов числами, а не строками текста
mapping = {label: int(label) for label in G.nodes()}
G = nx.relabel_nodes(G, mapping)

# каждому узлу добавили атрибут 'costs', отражающий затраты на привлечение каждого узла
attrs = {node: contract_cost_per_neighbor * G.degree(node) for node in G.nodes()}
nx.set_node_attributes(G, attrs, name='costs')

# каждому узлу добавили атрибут 'status', отражающий пользуется ли потребитель продуктом или нет ("заражен" или нет)
attrs = {node: False for node in G.nodes()}
nx.set_node_attributes(G, attrs, name='status')

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1CgegcarUDFN8jikjdtCga2jZ93NB99nL
To: /content/marketing_edges.txt
100% 817k/817k [00:00<00:00, 122MB/s]


In [ ]:
# базовая функция визуализации
def show_graphs(graphs):
    for i, graph in enumerate(graphs):    #0 - title, 1 - x axis name, 2 - y axis name, 3 - x axis series, 4 - [series names], 5 - [series]
        fig = go.Figure()
        for j in range(len(graph[4])): fig.add_trace(go.Scatter(y = graph[5][j], x = graph[3], mode = 'lines', name = graph[4][j]))
        fig.update_layout(template = 'plotly_white',
                                  title = graph[0],
                                  xaxis = dict(title = graph[1]),
                                  yaxis = dict(title = graph[2]),
                                  width = 850, height = 500)
        fig.show()

# функция отрисовки доступного бюджета маркетинговой компании по дням
def balance_graph(G, steps):
    balance_dict = balances(G, steps)
    show_graphs([
        ('Balance Dynamic', 'Day', 'Balance', np.arange(campaign_duration), ['balance'], [list(balance_dict.values())])
    ])
    print(f'Profit = {list(balance_dict.values())[-1] - initial_budget}')
    return balance_dict

# функция получения доступного бюджета маркетинговой компании по дням
def balances(G, steps):
    copy_G, balance_dict = G.copy(), {day: 0 for day in range(campaign_duration)}

    balance_dict[0] = balance = initial_budget
    for day in range(campaign_duration):
        balance += income_per_affected * len(update_affected(copy_G))
        if day in steps.keys():
            balance -= sum(copy_G.nodes[node]['costs'] for node in steps[day])
            if balance < 0: print('WARNING: Balance < 0')
            nx.set_node_attributes(copy_G, {node: True for node in steps[day]}, 'status')

        balance_dict[day] = balance
    return balance_dict

# функция просчета новых зараженных узлов на 1 день вперед (на одну итерацию вперед)
def update_affected(G):
    non_affected, new_affected = [node for node, attrs in G.nodes(data=True) if attrs.get('status') is False], []
    statuses = nx.get_node_attributes(G, 'status')
    for node in non_affected:
        neighbors = list(G.neighbors(node))
        affected_neighbors = sum(statuses[n] for n in neighbors)
        if affected_neighbors / len(neighbors) >= threshold:
            G.nodes[node]['status'] = True
            new_affected.append(node)
    return new_affected


In [ ]:
# ваш план маркетинговой компании в формате словаря
# день: [список id узлов, у которых закупается реклама]
marketing_campaign_steps = {
    0: [1, 2],
    1: [117]
}

# визуализация доступных для распределения средств вашей маркетинговой кампании по дням (включая изначальный бюджет), а также подсчет прибыли
balance_graph(G, marketing_campaign_steps)

Profit = -8900


{0: 2500,
 1: 1000,
 2: 1050,
 3: 1100,
 4: 1100,
 5: 1100,
 6: 1100,
 7: 1100,
 8: 1100,
 9: 1100,
 10: 1100,
 11: 1100,
 12: 1100,
 13: 1100,
 14: 1100,
 15: 1100,
 16: 1100,
 17: 1100,
 18: 1100,
 19: 1100,
 20: 1100,
 21: 1100,
 22: 1100,
 23: 1100,
 24: 1100,
 25: 1100,
 26: 1100,
 27: 1100,
 28: 1100,
 29: 1100,
 30: 1100,
 31: 1100,
 32: 1100,
 33: 1100,
 34: 1100,
 35: 1100,
 36: 1100,
 37: 1100,
 38: 1100,
 39: 1100,
 40: 1100,
 41: 1100,
 42: 1100,
 43: 1100,
 44: 1100,
 45: 1100,
 46: 1100,
 47: 1100,
 48: 1100,
 49: 1100,
 50: 1100,
 51: 1100,
 52: 1100,
 53: 1100,
 54: 1100,
 55: 1100,
 56: 1100,
 57: 1100,
 58: 1100,
 59: 1100}